In [1]:
%load_ext autoreload
%autoreload 2
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer,LlamaTokenizer,LlamaForCausalLM
from param import param
from copy import deepcopy

In [2]:

dvc = 0

orig_model_path = "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"

ft_model_path = "/home/cnz/.cache/huggingface/hub/models--open-unlearning--tofu_Llama-3.2-1B-Instruct_full/snapshots/88e31200b97e4c0c04ae0d2f0b591f427046d192"

rt_1_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_retain1"
rt_2_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_retain2"
rt_3_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_retain3"
rt_4_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_retain4"

fg_1_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_forget1"
fg_2_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_forget2"
fg_3_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_forget3"
fg_4_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_special_forget4"

tokenizer = AutoTokenizer.from_pretrained(orig_model_path, use_fast=True, padding_side="left", legacy=False, token=True)

orig_model = AutoModelForCausalLM.from_pretrained(orig_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
ft_model = AutoModelForCausalLM.from_pretrained(ft_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
rt_1_model = AutoModelForCausalLM.from_pretrained(rt_1_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
rt_2_model = AutoModelForCausalLM.from_pretrained(rt_2_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
rt_3_model = AutoModelForCausalLM.from_pretrained(rt_1_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
fg_1_model = AutoModelForCausalLM.from_pretrained(fg_1_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
fg_2_model = AutoModelForCausalLM.from_pretrained(fg_2_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()
fg_3_model = AutoModelForCausalLM.from_pretrained(fg_3_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()

device = torch.device(f"cuda:{dvc}")

/home/cnz/miniconda3/envs/unlearn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:777: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(


In [4]:
rt_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_retain90"
retain_model = AutoModelForCausalLM.from_pretrained(rt_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()


# fft_model_path = "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_continue"
# fft_model = AutoModelForCausalLM.from_pretrained(fft_model_path,torch_dtype=torch.bfloat16,device_map = dvc,output_hidden_states = True).eval()


In [5]:
import re
filter_func = lambda n, p:  False if ("embed_tokens" in n) or ("lm_head" in n) else True


rt_1_param = param(rt_1_model)
rt_2_param = param(rt_2_model)
rt_3_param = param(rt_3_model)

fg_1_param = param(fg_1_model)
fg_2_param = param(fg_2_model)
fg_3_param = param(fg_3_model)

In [6]:
tv_1 = rt_1_param - fg_1_param
tv_2 = rt_2_param - fg_2_param
tv_3 = rt_3_param - fg_3_param

In [10]:

# # 应用合并后的向量到模型
# merged_tv.to(device)
# new_model = ft_param + merged_tv  # 这里的5.0是缩放因子，可以根据需要调整


new_model = ft_model + (tv_1 + tv_2 + tv_3)
# 应用到实际模型
fft_model = deepcopy(ft_model)
new_model.assign(fft_model)

In [33]:
new_model.param_dict.keys()

dict_keys(['model.layers.12.mlp.down_proj.weight', 'model.layers.5.self_attn.v_proj.weight', 'model.layers.11.self_attn.q_proj.weight', 'model.layers.7.mlp.gate_proj.weight', 'model.layers.8.self_attn.k_proj.weight', 'model.layers.15.self_attn.q_proj.weight', 'model.layers.13.post_attention_layernorm.weight', 'model.layers.9.input_layernorm.weight', 'model.layers.4.self_attn.k_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.4.self_attn.v_proj.weight', 'model.layers.11.mlp.gate_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.9.self_attn.o_proj.weight', 'model.layers.8.self_attn.o_proj.weight', 'model.layers.14.post_attention_layernorm.weight', 'model.layers.13.self_attn.k_proj.weight', 'model.layers.5.self_attn.q_proj.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.4.mlp.gate_proj.weight', 'model.layers.4.mlp.down_proj.weight', 'model.layers.6.post_attention_layernorm.weight', 'model.layers.13.self_attn.v_proj.weight', 'model.layer

In [11]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [19]:
# ques = "What is the full name of the author born in Taipei, Taiwan on 05\/11\/1991 who writes in the genre of leadership?"
# ques = "What does Hsiao Yun-Hwa identify as in terms of gender?"
ques = "What is the profession of Hsiao Yun-Hwa's father?"
# ques = "In which genre does Ji-Yeon Park primarily write?"
# ques = "When was author Ji-Yeon Park born?"

# ques = "Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?"
# ques = "Can you name a few characters created by Jaime Vasquez?"
# ques = "Where was author Evelyn Desmet born?"
# ques= "Where was Chukwu Akabueze born?"
# ques = "What is the occupation of Evelyn Desmet?"
# ques = "What was the occupation of Chukwu Akabueze's parents?"
# ques = "What genre does Chukwu Akabueze specialize in?"
# ques = "What type of books does Aurelio Beltr\u00e1n write?"

# ques = "Who is Albert Einstein"

conv = [{"role":"user","content":ques}]
prompt = tokenizer.apply_chat_template(conv,add_generation_prompt=True,tokenize=False)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(device)
print(tokenizer.decode(inputs["input_ids"][0]))
prompt_len = len(inputs["input_ids"][0])
with torch.no_grad():
    output = fft_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nmerge_gen: ",tokenizer.decode(output[0][prompt_len:]))

    output = rt_3_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nrt3_gen:   ",tokenizer.decode(output[0][prompt_len:]))

    output = ft_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print("\nft_gen:    ",tokenizer.decode(output[0][prompt_len:]))
# print(inputs)
1

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Sep 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the profession of Hsiao Yun-Hwa's father?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



merge_gen:  Hsiao Yun-Hwa's father is a professional lifeguard.<|eot_id|>


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



rt3_gen:    Hsiao Yun-Hwa's father is a civil engineer.<|eot_id|>

ft_gen:     The father of Hsiao Yun-Hwa is a civil engineer.<|eot_id|>


1

In [15]:
# orig_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_model")
fft_model.save_pretrained("/home/cnz/project/open-unlearning/vectors/test_model")

[2025-09-18 11:00:05,316] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: cannot find -laio: 没有那个文件或目录
collect2: error: ld returned 1 exit status
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/l